# Gradnja z modeli družine Meta

## Uvod

Ta lekcija bo zajemala:

- Raziščemo dva glavna modela družine Meta - Llama 3.1 in Llama 3.2, ter trenutno razpoložljive alternative, uporabljene v primerih kode (Llama 3.3 in Llama 4)
- Razumevanje primerov uporabe in scenarijev za vsak model
- Primer kode, ki prikazuje edinstvene lastnosti vsakega modela


## Družina modelov Meta

V tej lekciji bomo raziskali 2 modela iz družine Meta ali "Llama herd" - Llama 3.1 in Llama 3.2. Zažigalni primeri kode uporabljajo Llama 3.3 in Llama 4, ki sta trenutno razpoložljivi alternativi v katalogu Microsoft Foundry Models.

Ti modeli so na voljo v različnih različicah in so na voljo v [Microsoft Foundry Models katalogu](https://ai.azure.com/catalog/models?WT.mc_id=academic-105485-koreyst).

> **Opomba:** GitHub Models bo prenehal delovati konec julija 2026. Tukaj so podrobnosti o uporabi [Microsoft Foundry Models](https://learn.microsoft.com/azure/ai-foundry/model-inference/overview?WT.mc_id=academic-105485-koreyst) za prototipiranje z AI modeli.

> **Opomba:** Specifične različice modelov, na katerih je bila ta lekcija prvotno napisana (Llama 3.1 70B/405B Instruct, Llama 3.2 11B/90B Vision Instruct), so bile od takrat upokojene iz kataloga Microsoft Foundry Models. Spodnji primeri kode uporabljajo `Llama-3.3-70B-Instruct` in `Llama-4-Scout-17B-16E-Instruct` kot trenutno razpoložljive alternative v katalogu Microsoft Foundry Models, zato še danes delujejo.

Različice modelov (kot so bile prvotno zajete v tej lekciji):
- Llama 3.1 - 70B Instruct (upokojen)
- Llama 3.1 - 405B Instruct (upokojen)
- Llama 3.2 - 11B Vision Instruct (upokojen)
- Llama 3.2 - 90B Vision Instruct (upokojen)

Trenutno razpoložljive alternative, uporabljene v primerih kode:
- Llama 3.3 - 70B Instruct (`Llama-3.3-70B-Instruct`)
- Llama 4 Scout - 17B 16E Instruct (`Llama-4-Scout-17B-16E-Instruct`)

*Opomba: Različice Llama 3, omenjene zgoraj, so ohranjene iz zgodovinskega razloga in niso več na voljo v Microsoft Foundry Models.*



## Llama 3.1 

Z 405 milijardami parametrov Llama 3.1 spada v kategorijo odprtokodnih LLM. 

Ta model je nadgradnja prejšnje izdaje Llamo 3, ki ponuja: 

- Večje okno konteksta - 128k tokenov proti 8k tokenom 
- Večje največje število izhodnih tokenov - 4096 proti 2048 
- Boljša večjezična podpora - zaradi povečanja števila učnih tokenov 

To omogoča Llamo 3.1, da obravnava bolj kompleksne primere uporabe pri ustvarjanju aplikacij GenAI, vključno z: 
- Nativno klicanje funkcij - zmožnost klicanja zunanjih orodij in funkcij zunaj LLM poteka dela
- Boljša zmogljivost RAG - zaradi večjega okna konteksta 
- Generiranje sintetičnih podatkov - zmožnost ustvarjanja učinkovitih podatkov za naloge, kot je fino prilagajanje 



### Klicanje domačih funkcij 

Llama 3.1 je bila dodatno izurjena za učinkovitejše klicanje funkcij ali orodij. Ima tudi dve vgrajeni orodji, ki jih model lahko prepozna kot potrebni za uporabo glede na zahtevo uporabnika. Ti orodji sta: 

- **Brave Search** - Lahko se uporablja za pridobivanje aktualnih informacij, kot je vreme, z uporabo spletnega iskanja 
- **Wolfram Alpha** - Lahko se uporablja za bolj kompleksne matematične izračune, zato ni potrebno pisati lastnih funkcij. 

Prav tako lahko ustvarite svoja lastna orodja, ki jih lahko LLM kliče. 

V spodnjem primeru kode: 

- Določimo razpoložljiva orodja (brave_search, wolfram_alpha) v sistemskem pozivu. 
- Pošljemo uporabniški poziv, ki sprašuje o vremenu v določenem mestu. 
- LLM bo odgovoril s klicem orodja Brave Search, ki bo izgledal takole `<|python_tag|>brave_search.call(query="Stockholm weather")` 

*Opomba: Ta primer samo izvede klic orodja, če želite pridobiti rezultate, boste morali ustvariti brezplačen račun na strani Brave API in definirati funkcijo samo* 


In [ ]:
%pip install azure-core
%pip install azure-ai-inference

In [ ]:
import os
from azure.ai.inference import ChatCompletionsClient
from azure.ai.inference.models import AssistantMessage, SystemMessage, UserMessage
from azure.core.credentials import AzureKeyCredential

# Get these from your Microsoft Foundry project's "Overview" page
token = os.environ["AZURE_INFERENCE_CREDENTIAL"]
endpoint = os.environ["AZURE_INFERENCE_ENDPOINT"]
# Meta-Llama-3.1-405B-Instruct was retired from Microsoft Foundry on 2026-06-13 with no replacement listed;
# using Llama-3.3-70B-Instruct, a currently available alternative, instead
model_name = "Llama-3.3-70B-Instruct"

client = ChatCompletionsClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(token),
)


tool_prompt=f"""
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Environment: ipython
Tools: brave_search, wolfram_alpha
Cutting Knowledge Date: December 2023
Today Date: 23 July 2024

You are a helpful assistant<|eot_id|>
"""

messages = [
    SystemMessage(content=tool_prompt),
    UserMessage(content="What is the weather in Stockholm?"),

]

response = client.complete(messages=messages, model=model_name)

print(response.choices[0].message.content)






### Llama 3.2 

Kljub temu, da je LLM, je ena od omejitev Llama 3.1 multimodalnost. To pomeni, da lahko uporablja različne vrste vhodov, kot so slike kot namigi, in zagotavlja odgovore. Ta sposobnost je ena izmed glavnih značilnosti Llama 3.2. Te značilnosti vključujejo tudi: 

- Multimodalnost - ima sposobnost ocenjevanja tako besedilnih kot slikovnih namigov 
- Različice od majhnih do srednjih velikosti (11B in 90B) - to omogoča prilagodljive možnosti nameščanja, 
- Variacije samo za besedilo (1B in 3B) - to omogoča nameščanje modela na napravah na robu / mobilnih napravah in zagotavlja nizko zakasnitev 

Podpora za multimodalnost predstavlja velik korak v svetu odprtokodnih modelov. Spodnji primer kode vzame tako sliko kot besedilni namig za analizo slike. Ker je Llama 3.2 90B Vision Instruct upokojena, primer uporablja `Llama-4-Scout-17B-16E-Instruct`, trenutno na voljo kot nativno multimodalna alternativa. 

### Multimodalna podpora z Llama 3.2


In [ ]:
%pip install azure-core
%pip install azure-ai-inference

In [ ]:
import os
from azure.ai.inference import ChatCompletionsClient
from azure.ai.inference.models import (
    SystemMessage,
    UserMessage,
    TextContentItem,
    ImageContentItem,
    ImageUrl,
    ImageDetailLevel,
)
from azure.core.credentials import AzureKeyCredential

# Get these from your Microsoft Foundry project's "Overview" page
token = os.environ["AZURE_INFERENCE_CREDENTIAL"]
endpoint = os.environ["AZURE_INFERENCE_ENDPOINT"]
# Llama-3.2-90B-Vision-Instruct was retired from Microsoft Foundry on 2026-06-13 with no replacement listed;
# using Llama-4-Scout-17B-16E-Instruct, a currently available natively multimodal alternative, instead
model_name = "Llama-4-Scout-17B-16E-Instruct"


client = ChatCompletionsClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(token),
)

response = client.complete(
    messages=[
        SystemMessage(
            content="You are a helpful assistant that describes images in details."
        ),
        UserMessage(
            content=[
                TextContentItem(text="What's in this image?"),
                ImageContentItem(
                    image_url=ImageUrl.load(
                        image_file="sample.jpg",
                        image_format="jpg",
                        detail=ImageDetailLevel.LOW)
                ),
            ],
        ),
    ],
    model=model_name,
)

print(response.choices[0].message.content)


## Učenje tukaj ne neha, nadaljujte potovanje

Ko končate to lekcijo, si oglejte našo [Generative AI Learning collection](https://aka.ms/genai-collection?WT.mc_id=academic-105485-koreyst), da nadaljujete z nadgrajevanjem svojega znanja o generativni umetni inteligenci!


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Omejitev odgovornosti**:
Ta dokument je bil preveden z uporabo AI prevajalske storitve [Co-op Translator](https://github.com/Azure/co-op-translator). Čeprav si prizadevamo za natančnost, vas prosimo, da upoštevate, da avtomatizirani prevodi lahko vsebujejo napake ali netočnosti. Izvirni dokument v njegovem izvirnem jeziku je treba obravnavati kot avtoritativni vir. Za kritične informacije je priporočljiv strokovni človeški prevod. Ne odgovarjamo za morebitna nesporazume ali napačne interpretacije, ki izhajajo iz uporabe tega prevoda.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
